# Wavelet Analysis: Theory and Application to African Easterly Waves

## 1. Introduction and Theory

In climate science, we frequently analyse time series data to find dominant cycles—such as the El Niño-Southern Oscillation (ENSO) or diurnal temperature variations. As we saw in the last lesson, Fourier analysis is the traditional tool for this, decomposing a signal into constituent sine and cosine waves to identify dominant frequencies.

However, the Fourier transform only allows us to find **stationary** signals, i.e. the resul is is an "average" measure of the power of each frequency over the entire timeseries.  In some cases, there may be a signal we want to detect which is . If we want to detect African Easterly Waves (AEWs)—synoptic disturbances that only occur during the West African Monsoon season—a standard Fourier transform will tell us a 3-5 day periodicity exists, but it cannot tell us *when* it occurs.

**Wavelet analysis** overcomes this by using localized wave-like functions (wavelets). Instead of infinite sine waves, wavelets decay to zero at their edges. By stretching (scaling) and shifting (translating) these wavelets across our time series, we build a two-dimensional picture of both frequency (scale) *and* time.

### The Continuous Wavelet Transform (CWT)

The CWT of a discrete time series $x_n$ with equal time spacing $\delta t$ is defined as the convolution of $x_n$ with a scaled and translated version of a mother wavelet, $\psi$:

$$ W_n(s) = \sum_{n'=0}^{N-1} x_{n'} \psi^* \left[ \frac{(n' - n)\delta t}{s} \right] $$

Where:
* $W_n(s)$ are the wavelet coefficients.
* $s$ is the wavelet scale (proportional to the period/frequency).
* $n$ is the localized time index.
* $\psi^*$ is the complex conjugate of the normalized wavelet function.

### The Morlet Wavelet

The most common "mother wavelet" used in geophysics is the Morlet wavelet. It is constructed by taking a complex sine wave and multiplying it by a Gaussian envelope, perfectly localizing it in time.

$$ \psi_0(\eta) = \pi^{-1/4} e^{i \omega_0 \eta} e^{-\eta^2/2} $$

Where $\eta$ is non-dimensional time, and $\omega_0$ is the non-dimensional frequency. In climate science, we almost always set $\omega_0 = 6$ because it yields a wavelet scale $s$ that is nearly identical to the Fourier period ($T \approx 1.03s$), making it highly intuitive to read.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import pywt

plt.rcParams['figure.figsize'] = (12, 6)

# 1. Define the Morlet Wavelet
def morlet_wavelet(t, w0=6.0):
    """Generates a complex Morlet wavelet."""
    return np.pi**(-0.25) * np.exp(1j * w0 * t) * np.exp(-0.5 * t**2)

# 2. Build the CWT from scratch
def cwt_scratch(data, dt, scales, w0=6.0):
    """
    Computes the Continuous Wavelet Transform via temporal convolution.
    """
    n = len(data)
    cwt_matrix = np.zeros((len(scales), n), dtype=complex)
    
    for i, s in enumerate(scales):
        # Create localized time vector for the wavelet (e.g., +/- 8 standard deviations)
        t_wavelet = np.arange(-8*s, 8*s, dt)
        
        # Generate scaled wavelet (1/sqrt(s) preserves energy variance across scales)
        wavelet = morlet_wavelet(t_wavelet / s, w0) * (1 / np.sqrt(s))
        
        # Convolve data with the complex conjugate of the wavelet
        cwt_matrix[i, :] = np.convolve(data, np.conj(wavelet), mode='same') * dt
        
    return cwt_matrix

## 2. Toy Example: Non-Stationary Sine Waves

Before tackling messy climate data, let's prove our code works. We will create a signal that is non-stationary: 
1. For the first 500 time steps, it has a period of 20 (high frequency).
2. For the last 500 time steps, it has a period of 80 (low frequency).

A traditional Fourier transform would just show peaks at 20 and 80, giving no information about *when* those frequencies occurred. Let's see what our wavelet scalogram does.

In [ ]:
# Generate toy data
t_toy = np.arange(1000)
signal = np.zeros(1000)
signal[:500] = np.sin(2 * np.pi * t_toy[:500] / 20)  # Period = 20
signal[500:] = np.sin(2 * np.pi * t_toy[500:] / 80)  # Period = 80

# Define scales to investigate
scales_toy = np.arange(1, 150, 2)

# Compute CWT
cwt_toy = cwt_scratch(signal, dt=1.0, scales=scales_toy)
power_toy = np.abs(cwt_toy)**2

# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

ax1.plot(t_toy, signal, color='black')
ax1.set_title('Toy Signal: High frequency shifting to Low frequency')
ax1.set_ylabel('Amplitude')
ax1.set_xlim(0, 1000)

im = ax2.imshow(power_toy, extent=[0, 1000, scales_toy[-1], scales_toy[0]], 
                aspect='auto', cmap='viridis', interpolation='bilinear')
ax2.invert_yaxis()
ax2.set_title('Wavelet Power Spectrum (Scalogram)')
ax2.set_ylabel('Period (Scale)')
ax2.set_xlabel('Time Step')
fig.colorbar(im, ax=ax2, orientation='horizontal', label='Wavelet Power')

plt.tight_layout()
plt.show()

Notice how clearly the scalogram identifies both the exact period of the waves and the exact time step (500) where the regime shifts!

---
## 3. Application: African Easterly Waves in ERA5 Data

Now we apply this to a real meteorological phenomenon. African Easterly Waves (AEWs) are westward-propagating synoptic disturbances that originate over North Africa. They are primary precursors to Atlantic tropical cyclones.

AEWs are best detected in the meridional (v-component) wind at the 700 hPa level, typically exhibiting periods of **3 to 5 days**. Because they rely on the African Easterly Jet, they are highly seasonal, peaking during the summer monsoon (July to September).

We will load ERA5 reanalysis data. *(Note: For this notebook to run cleanly out-of-the-box, we generate a synthetic `xarray.DataArray` that mimics the structure and statistical properties of an ERA5 V-wind slice at 15°N, 0°E over a full year).* Let's use our CWT to detect the summer AEW activity.

In [ ]:
# 1. Create a simulated ERA5 xarray dataset for the year 2020
time_era5 = pd.date_range(start="2020-01-01", end="2020-12-31", freq="D")
np.random.seed(100)

# Base noise (winter/spring/fall)
v_wind_data = np.random.normal(0, 2.0, size=len(time_era5))

# Inject AEW activity (3-5 day periods) specifically during JJAS (June-Sept)
jjas_mask = (time_era5.month >= 6) & (time_era5.month <= 9)
aew_signal = 5.0 * np.sin(2 * np.pi * np.arange(len(time_era5)) / 4.2) # 4.2 day wave
v_wind_data[jjas_mask] += aew_signal[jjas_mask] * np.hanning(jjas_mask.sum()) # Apply envelope

# Pack into xarray
da_vwind = xr.DataArray(
    v_wind_data,
    coords={"time": time_era5},
    dims=["time"],
    name="v",
    attrs={"long_name": "Meridional wind component", "units": "m/s", "level": "700hPa"}
)

# 2. Perform Wavelet Analysis on the ERA5 Data
scales_era5 = np.arange(1, 30, 0.5) # Look at periods from 1 to 30 days
cwt_era5 = cwt_scratch(da_vwind.values, dt=1.0, scales=scales_era5)
power_era5 = np.abs(cwt_era5)**2

# 3. Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={'height_ratios': [1, 2]})

da_vwind.plot(ax=ax1, color='teal')
ax1.set_title('ERA5 700 hPa Meridional Wind Anomaly (15°N, 0°E)')
ax1.set_ylabel('V-wind (m/s)')
ax1.grid(True, linestyle=':', alpha=0.6)

time_plot = da_vwind.time.values
im2 = ax2.imshow(power_era5, extent=[0, len(time_plot), scales_era5[-1], scales_era5[0]], 
                 aspect='auto', cmap='magma', interpolation='bilinear')
ax2.invert_yaxis()
ax2.set_title('Wavelet Power Spectrum: Detecting the AEW Season')
ax2.set_ylabel('Period (Days)')
ax2.set_xlabel('Day of Year (2020)')
fig.colorbar(im2, ax=ax2, orientation='horizontal', label='Wavelet Power')

# Highlight AEW Band
ax2.axhline(y=3, color='white', linestyle='--', alpha=0.5)
ax2.axhline(y=5, color='white', linestyle='--', alpha=0.5)
ax2.text(10, 4, 'AEW Band (3-5 days)', color='white', verticalalignment='center')

plt.tight_layout()
plt.show()

## 4. Transitioning to Standard Packages (`PyWavelets`)

While writing the code from scratch demystifies the mathematics, doing this operation manually using standard convolution is slow for large datasets (like 40 years of daily global gridded data). 

Professional packages utilize the **Fast Fourier Transform (FFT)** to compute the convolution theorem rapidly in frequency space. They also properly handle zero-padding at the boundaries to prevent circular convolution wrap-around.

Here, we implement the exact same analysis using `PyWavelets` (`pywt`).

In [ ]:
# In PyWavelets, the complex Morlet is 'cmorB-C', where B is bandwidth and C is center frequency.
# To match our analytical w0=6, we use a center frequency of ~0.955 (since pywt defines it as w0/(2*pi))
wavelet_pkg = 'cmor1.0-0.955'

# pywt.cwt returns the coefficients and the corresponding frequencies
cwt_pkg, freqs_pkg = pywt.cwt(da_vwind.values, scales_era5, wavelet_pkg, sampling_period=1.0)
power_pkg = np.abs(cwt_pkg)**2

# Compare the two arrays to prove they are computing the same thing
diff = np.nanmean(np.abs(power_era5 - power_pkg))
print(f"Mean Absolute Difference between bespoke code and PyWavelets: {diff:.4f}")

## 5. The Cone of Influence (COI)

If you look closely at the edges of our previous scalograms, you might notice the power drops off or behaves strangely. This introduces a critical concept in wavelet analysis: **The Cone of Influence (COI)**.

Because the wavelet is computed using a convolution, it requires data on both sides of a given time step. As we approach the beginning or end of our time series, the scaled wavelet extends beyond the bounds of our actual data. 

To compute the transform at the edges, algorithms implicitly pad the data (usually with zeros). This zero-padding reduces the variance at the edges, creating artificially low wavelet power. 

The COI defines the region of the wavelet spectrum where these edge effects become significant. It is mathematically defined as the e-folding time of the autocorrelation of the wavelet power. For a Morlet wavelet with $\omega_0 = 6$, the e-folding time is roughly:

$$ t_{COI} = \sqrt{2} s $$

Where $s$ is the wavelet scale. Because lower frequencies (larger scales) use wider wavelets, the edge effects penetrate deeper into the time series. This creates a cone-shaped region at the boundaries. Any wavelet power outside the COI (often shaded out) should be interpreted with high skepticism.

In [ ]:
# Calculate the Cone of Influence (COI)
# For Morlet w0=6, the e-folding time is sqrt(2) * scale
coi_boundary = np.sqrt(2) * scales_era5

# Create the plot again using the PyWavelets power
plt.figure(figsize=(12, 6))
plt.imshow(power_pkg, extent=[0, len(time_plot), scales_era5[-1], scales_era5[0]], 
           aspect='auto', cmap='magma', interpolation='bilinear')
plt.gca().invert_yaxis()

# Plot the COI boundaries as dashed lines
plt.plot(coi_boundary, scales_era5, color='white', linestyle='--', linewidth=2)
plt.plot(len(time_plot) - coi_boundary, scales_era5, color='white', linestyle='--', linewidth=2)

# Shade the regions outside the COI to indicate they are subject to edge effects
plt.fill_betweenx(scales_era5, 0, coi_boundary, color='white', alpha=0.4, hatch='//')
plt.fill_betweenx(scales_era5, len(time_plot) - coi_boundary, len(time_plot), color='white', alpha=0.4, hatch='\\\\')

plt.colorbar(label='Wavelet Power')
plt.title('Wavelet Power Spectrum with Cone of Influence (COI)')
plt.xlabel('Day of Year (2020)')
plt.ylabel('Period (Days)')

# Highlight the 3-5 day AEW band again
plt.axhline(y=3, color='cyan', linestyle=':', alpha=0.8)
plt.axhline(y=5, color='cyan', linestyle=':', alpha=0.8)

plt.show()